In [1]:
from Functions import utils as u

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:


data = u.extract_data(UMP_threshold=-3.75)

display(data)

#features_and_label = ['Teff', 'logg', 'NUVDered', 'colorColorY']
features_and_label = ['Teff', 'bp_rp_dered', 'colorColorY','UMP_flag'] #add logg once we figure out NaNs

selected_data = data[features_and_label].copy()


,Survey,FeH,NUVDered,bp_rp_dered,Teff,logg,colorColorY,UMP_flag
0,Bonifacio,-2.83,20.194627,1.003094,6303,4.25,-1.016775,0
1,Lai,-3.00,21.265522,0.737908,4805,1.48,-0.965118,0
2,Lai,-3.50,21.244102,0.775386,4750,1.31,-0.109841,0
3,Lai,-2.90,20.176450,1.003778,5163,2.47,-0.312304,0
4,Lai,-3.95,22.584123,1.754697,4827,1.51,-8.287684,1
...,...,...,...,...,...,...,...,...
485,Yong,-2.60,18.570030,0.582178,6507,NaN,-1.297989,0
486,Yong,-2.65,17.412597,0.570395,6624,NaN,-1.041163,0
487,Yong,-2.60,18.019629,0.580172,6479,NaN,-1.252573,0
488,Yong,-2.68,18.129256,0.664409,6196,NaN,-1.605187,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


def scale_and_split(data, target_column, test_size=0.2, random_state=42):
    """
    Splits the dataset into training and validation sets, and scales the features.

    Parameters:
    - data: pd.DataFrame
    - target_column: str, name of the target column
    - test_size: float, proportion of validation set
    - random_state: int, for reproducibility

    Returns:
    - X_train_scaled, X_val_scaled, y_train, y_val, scaler
    """
    # Separate features and target
    X = data.drop(columns=[target_column]).copy()
    y = data[target_column].copy()

    # Split into train and validation
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    return X_train_scaled, X_val_scaled, y_train, y_val, scaler

x_train, x_val, y_train, y_val, scaler = scale_and_split(selected_data, 'UMP_flag',)



In [4]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

def tune_logistic_regression(X_train, X_val, y_train, y_val, C_values=None):
    """
    Trains logistic regression models over a range of C values and plots accuracy.

    Parameters:
    - X_train, X_val: scaled feature arrays
    - y_train, y_val: target arrays
    - C_values: list or array of regularization strengths to try

    Returns:
    - best_model: trained LogisticRegression instance with best C
    """
    if C_values is None:
        C_values = np.logspace(-3, 2, 10)

    accuracies = []
    models = []

    for C in C_values:
        model = LogisticRegression(C=C, solver='liblinear', max_iter=1000)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        accuracies.append(acc)
        models.append(model)

    # Plot accuracy vs C
    plt.figure(figsize=(8, 5))
    plt.semilogx(C_values, accuracies, marker='o')
    plt.xlabel("Regularization Strength (C)")
    plt.ylabel("Validation Accuracy")
    plt.title("Logistic Regression Hyperparameter Tuning")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Best model
    best_idx = np.argmax(accuracies)
    best_model = models[best_idx]
    print(f"Best C: {C_values[best_idx]:.4f} | Accuracy: {accuracies[best_idx]:.4f}")

    # Confusion matrix
    y_pred_best = best_model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred_best)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title("Confusion Matrix (Best Model)")
    plt.show()

    return best_model
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

def tune_logistic_regression(X_train, X_val, y_train, y_val, C_values=None, patience= 3):
    """
    Trains logistic regression models over a range of C values and plots accuracy.

    Parameters:
    - X_train, X_val: scaled feature arrays
    - y_train, y_val: target arrays
    - C_values: list or array of regularization strengths to try

    Returns:
    - best_model: trained LogisticRegression instance with best C
    """
    if C_values is None:
        C_values = np.logspace(-4, 2, 20)

    best_acc = 0
    best_model = None
    wait = 0
    accuracies = []
    models =[]

    for C in C_values:
        model = LogisticRegression(C=C, solver='liblinear', max_iter=1000)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        acc = accuracy_score(y_val, y_pred)
        accuracies.append(acc)
        models.append(model)

        if acc > best_acc:
            best_acc = acc
            best_model = model
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered at C={C:.4f} for regularization tuning.")
                break


    # Plot accuracy vs C
    plt.figure(figsize=(8, 5))
    plt.semilogx(C_values, accuracies, marker='o')
    plt.xlabel("Regularization Strength (C)")
    plt.ylabel("Validation Accuracy")
    plt.title("Logistic Regression Hyperparameter Tuning")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Best model
    best_idx = np.argmax(accuracies)
    best_model = models[best_idx]
    print(f"Best C: {C_values[best_idx]:.4f} | Accuracy: {accuracies[best_idx]:.4f}")

    # Confusion matrix
    y_pred_best = best_model.predict(X_val)
    cm = confusion_matrix(y_val, y_pred_best)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap='Blues')
    plt.title("Confusion Matrix (Best Model)")
    plt.show()

    return best_model



In [5]:
# Assuming you already have X_train_scaled, X_val_scaled, y_train, y_val
best_model = tune_logistic_regression(x_train, x_val, y_train, y_val)


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values